# Conceptos Fundamentales — LLMs y Generative AI

**Módulo 3 — Introducción a AI Engineering** | DSRP Machine Learning Engineering

---

> 📖 Este notebook es un **glosario de referencia** para los conceptos fundamentales de LLMs, Transformers y GenAI. Cada concepto incluye definición, matemática, visualización y código.

### Índice
1. [Tokens y Tokenización](#1)
2. [Embeddings — representación vectorial](#2)
3. [Similitud coseno y distancia euclidiana](#3)
4. [Softmax — de logits a probabilidades](#4)
5. [Positional Encodings — codificar el orden](#5)
6. [Attention Mechanism — Query, Key, Value](#6)
7. [Multi-Head Attention](#7)
8. [Feed-Forward Networks (FFN)](#8)
9. [Layer Normalization y conexiones residuales](#9)
10. [Autoregresión — generación token a token](#10)
11. [Sampling strategies — temperature, top-k, top-p](#11)
12. [Perplexity — métrica de evaluación](#12)
13. [Context Window y KV Cache](#13)
14. [Fine-tuning vs Prompting](#14)
15. [Scaling Laws — más grande = mejor](#15)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cosine, euclidean
from scipy.special import softmax

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
np.set_printoptions(precision=3, suppress=True)


---
<a id='1'></a>
## 1. Tokens y Tokenización

**Definición:** La **tokenización** es el proceso de dividir texto en unidades pequeñas llamadas **tokens**. Un token puede ser una palabra completa, una subpalabra, un carácter o incluso un byte.

### ¿Por qué no usar palabras completas?

| Problema | Solución con subpalabras |
|---|---|
| Vocabulario gigante (millones de palabras) | Vocabulario manejable (30k-200k tokens) |
| Palabras fuera del vocabulario (OOV) | Cualquier palabra se puede representar |
| No maneja morfología (correr, corriendo, corrió) | Comparte raíz: `[corr][iendo]`, `[corr][ió]` |

### Algoritmos principales

**1. Byte-Pair Encoding (BPE)** — usado por GPT, LLaMA
- Empieza con caracteres individuales
- Iterativamente fusiona los pares más frecuentes
- Ejemplo: `"running"` → `["run", "ning"]` o `["runn", "ing"]` según frecuencia

**2. WordPiece** — usado por BERT
- Similar a BPE pero maximiza la verosimilitud del corpus
- Usa `##` para marcar continuaciones: `["run", "##ning"]`

**3. SentencePiece** — usado por T5, LLaMA
- Trata el texto como secuencia de bytes (no requiere pre-tokenización)
- Soporta cualquier idioma sin espacios (chino, japonés)

### Propiedades importantes

- **Vocabulario fijo:** típicamente 30k-200k tokens
- **Reversible:** tokens → texto original (casi siempre)
- **Dependiente del modelo:** cada modelo tiene su propio tokenizer
- **Impacto en costo:** más tokens = más caro (APIs cobran por token)

In [ ]:
# Simulación de tokenización BPE simplificada
texto = "running runner run"
print(f"Texto original: '{texto}'\\n")

# Paso 1: caracteres individuales
tokens_iniciales = list(texto.replace(" ", "_"))
print(f"Paso 1 — caracteres: {tokens_iniciales}\\n")

# Paso 2: fusionar pares frecuentes (simulado manualmente)
# En BPE real, se cuenta frecuencia y se fusiona iterativamente
vocab_bpe = {
    "r": "r", "u": "u", "n": "n", "i": "i", "g": "g", "e": "e",
    "run": "run", "ning": "ning", "ner": "ner", "_": "_",
}

# Tokenización final (simulada)
tokens_finales = ["run", "ning", "_", "run", "ner", "_", "run"]
print(f"Paso 2 — después de fusiones BPE: {tokens_finales}\\n")

# Cada token tiene un ID en el vocabulario
vocab_ids = {tok: i for i, tok in enumerate(sorted(vocab_bpe.keys()))}
ids = [vocab_ids[t] for t in tokens_finales]
print(f"IDs en el vocabulario: {ids}")
print(f"\\nVocabulario completo ({len(vocab_ids)} tokens): {vocab_ids}")

# Visualización: comparación de tamaños de vocabulario
fig, ax = plt.subplots(figsize=(8, 4))
metodos = ['Palabras\\ncompletas', 'Caracteres', 'BPE/WordPiece\\n(subpalabras)']
tamaños = [500000, 256, 50000]
colores = ['tomato', 'steelblue', 'green']
bars = ax.barh(metodos, tamaños, color=colores, alpha=0.8)
for bar, tam in zip(bars, tamaños):
    ax.text(tam + 10000, bar.get_y() + bar.get_height()/2,
            f'{tam:,}', va='center', fontweight='bold')
ax.set_xlabel('Tamaño del vocabulario')
ax.set_title('Comparación de métodos de tokenización')
ax.set_xscale('log')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout(); plt.show()


---
<a id='1'></a>
## 1. Tokens y Tokenización

**Definición:** La **tokenización** es el proceso de dividir texto en unidades pequeñas llamadas **tokens**. Un token puede ser una palabra completa, una subpalabra, un carácter o incluso un byte.

### ¿Por qué no usar palabras completas?

| Problema | Solución con subpalabras |
|---|---|
| Vocabulario gigante (millones de palabras) | Vocabulario manejable (30k-200k tokens) |
| Palabras fuera del vocabulario (OOV) | Cualquier palabra se puede representar |
| No maneja morfología (correr, corriendo, corrió) | Comparte raíz: `[corr][iendo]`, `[corr][ió]` |

### Algoritmos principales

**1. Byte-Pair Encoding (BPE)** — usado por GPT, LLaMA
- Empieza con caracteres individuales
- Iterativamente fusiona los pares más frecuentes
- Ejemplo: `"running"` → `["run", "ning"]` o `["runn", "ing"]` según frecuencia

**2. WordPiece** — usado por BERT
- Similar a BPE pero maximiza la verosimilitud del corpus
- Usa `##` para marcar continuaciones: `["run", "##ning"]`

**3. SentencePiece** — usado por T5, LLaMA
- Trata el texto como secuencia de bytes (no requiere pre-tokenización)
- Soporta cualquier idioma sin espacios (chino, japonés)

### Propiedades importantes

- **Vocabulario fijo:** típicamente 30k-200k tokens
- **Reversible:** tokens → texto original (casi siempre)
- **Dependiente del modelo:** cada modelo tiene su propio tokenizer
- **Impacto en costo:** más tokens = más caro (APIs cobran por token)